<a href="https://colab.research.google.com/github/jungeun1228/2026_08_Tahoe-100M_scRNA-seq-analysis/blob/main/00_Tahoe_Data_Download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tahoe-100M dataset download
Selected conditions:
17 compounds
3 concentrations: 0.05, 0.5, 5 µM
5 cell lines: HT-29, PANC-1, HepG2/C3A, A-172, BT-474
Plate-matched DMSO controls included

In [ ]:
# Check RAN and disk
import psutil
import shutil

ram = psutil.virtual_memory()
disk = shutil.disk_usage("/content")

print(f"RAM: {ram.total / 1024**3:.1f} GB")
print(f"Available RAM: {ram.available / 1024**3:.1f} GB")
print(f"Free disk: {disk.free / 1024**3:.1f} GB")

RAM: 51.0 GB
Available RAM: 49.8 GB
Free disk: 205.5 GB


In [ ]:
%pip install -q -U duckdb huggingface_hub pyarrow pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 131.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


In [ ]:
# Connect to google drive
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/Tahoe_project"

os.makedirs(f"{PROJECT}/raw", exist_ok=True)
os.makedirs(f"{PROJECT}/checkpoints", exist_ok=True)

print("Project folder ready!")

Project folder ready!


In [ ]:
# Step 1: Import sample list to colab

from google.colab import files

uploaded = files.upload()

Saving selected_samples.csv to selected_samples.csv


In [ ]:
import os
import shutil
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Tahoe_project")

PARTS = PROJECT / "raw_parts"
CHECKPOINTS = PROJECT / "checkpoints"

PARTS.mkdir(parents=True, exist_ok=True)
CHECKPOINTS.mkdir(parents=True, exist_ok=True)

# Upload한 CSV를 Drive에 영구 저장
csv_path = PROJECT / "selected_samples.csv"

if not csv_path.exists():
    shutil.copy2(
        "/content/selected_samples.csv",
        csv_path
    )

samples = pd.read_csv(csv_path)

print("Samples:", len(samples))
print(samples.head())

Samples: 85
     sample   plate                          drug  concentration unit  \
0  smp_1576  plate1  Methylprednisolone succinate           0.05   uM   
1  smp_1672  plate2  Methylprednisolone succinate           0.50   uM   
2  smp_1768  plate3  Methylprednisolone succinate           5.00   uM   
3  smp_1784  plate4     Chlorhexidine (diacetate)           0.05   uM   
4  smp_1824  plate4                        Thymol           0.05   uM   

   is_control  
0       False  
1       False  
2       False  
3       False  
4       False  


In [ ]:
# Step 2: Tahoe download setting
import duckdb
import json
from huggingface_hub import HfApi, HfFileSystem, hf_hub_download

REPO = "tahoebio/Tahoe-100M"

api = HfApi()

# 데이터 버전을 고정해서 세션이 달라져도 동일한 파일 사용
revision_file = PROJECT / "source_revision.txt"

if revision_file.exists():
    revision = revision_file.read_text().strip()
else:
    revision = api.repo_info(
        REPO,
        repo_type="dataset"
    ).sha

    revision_file.write_text(revision)

# 전체 expression shard 목록
manifest_file = PROJECT / "source_shards.json"

if manifest_file.exists():
    shards = json.loads(manifest_file.read_text())
else:
    repo_files = api.list_repo_files(
        REPO,
        repo_type="dataset",
        revision=revision
    )

    shards = sorted([
        f for f in repo_files
        if f.startswith("data/train-")
        and f.endswith(".parquet")
    ])

    manifest_file.write_text(json.dumps(shards))

assert len(shards) > 0, "Expression shards not found"

print("Dataset revision:", revision)
print("Expression shards:", len(shards))

# 필요한 gene metadata도 미리 저장
gene_meta = hf_hub_download(
    repo_id=REPO,
    repo_type="dataset",
    filename="metadata/gene_metadata.parquet",
    revision=revision
)

gene_dest = PROJECT / "gene_metadata.parquet"

if not gene_dest.exists():
    shutil.copy2(gene_meta, gene_dest)

# DuckDB 연결
con = duckdb.connect()

con.register_filesystem(HfFileSystem())

con.execute("SET memory_limit = '16GB'")
con.execute("SET threads = 2")
con.execute("SET preserve_insertion_order = false")

print("Ready!")

Dataset revision: 2dc57900b7981cfcf5e211527169a0b006546a95
Expression shards: 3388


metadata/gene_metadata.parquet: reconstructing file:   0%|          |  0.00B / 1.33MB            

metadata/gene_metadata.parquet: downloading bytes:           |  0.00B            

Ready!


In [ ]:
# Step 3: Function for download
CELL_IDS = [
    "CVCL_0320",  # HT-29
    "CVCL_0480",  # PANC-1
    "CVCL_1098",  # HepG2/C3A
    "CVCL_0131",  # A-172
    "CVCL_0179",  # BT-474
]

import pyarrow.parquet as pq

SAMPLE_IDS = samples["sample"].dropna().astype(str).unique().tolist()

def sql_quote(x):
    return "'" + str(x).replace("'", "''") + "'"

sample_filter = ", ".join(map(sql_quote, SAMPLE_IDS))
cell_filter = ", ".join(map(sql_quote, CELL_IDS))

TMP = Path("/content/tahoe_tmp")
TMP.mkdir(exist_ok=True)


def download_batch(batch_files, batch_name):

    output = PARTS / f"{batch_name}.parquet"
    checkpoint = CHECKPOINTS / f"{batch_name}.json"
    temp_file = TMP / f"{batch_name}.parquet"

    # 이미 완료한 batch는 건너뛰기
    if checkpoint.exists():
        print(f"Skipping {batch_name}")
        return

    urls = [
        f"hf://datasets/{REPO}@{revision}/{f}"
        for f in batch_files
    ]

    url_sql = "[" + ", ".join(map(sql_quote, urls)) + "]"

    query = f"""
    COPY (
        SELECT
            BARCODE_SUB_LIB_ID AS cell_id,
            sample,
            drug,
            plate,
            cell_line_id,
            "moa-fine" AS moa_fine,
            genes,
            expressions

        FROM read_parquet({url_sql})

        WHERE sample IN ({sample_filter})
          AND cell_line_id IN ({cell_filter})

    )
    TO {sql_quote(str(temp_file))}
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """

    print(f"Downloading: {batch_name}", flush=True)

    # 이전 실패에서 남은 임시 파일 제거
    if temp_file.exists():
        temp_file.unlink()

    con.execute(query)

    # 실제 저장된 cell 수 확인
    n_cells = pq.ParquetFile(temp_file).metadata.num_rows

    # 데이터가 존재하는 batch만 Drive에 보관
    if n_cells > 0:

        shutil.copy2(temp_file, output)

        # Drive에 정상적으로 복사됐는지 검증
        verified_cells = pq.ParquetFile(output).metadata.num_rows

        if verified_cells != n_cells:
            raise RuntimeError("File verification failed")

    elif output.exists():
        output.unlink()

    # 성공한 batch만 완료 기록 생성
    checkpoint.write_text(json.dumps({
        "batch": batch_name,
        "shards": batch_files,
        "n_cells": n_cells,
        "revision": revision
    }))

    temp_file.unlink()

    print(
        f"Completed: {batch_name} | "
        f"Cells: {n_cells:,}",
        flush=True
    )

In [ ]:
# Step 4: check the first shard
download_batch(
    shards[:1],
    "batch_0000"
)

Downloading: batch_0000
Completed: batch_0000 | Cells: 147


In [ ]:
# Step 5: proceed in batch
BATCH_SIZE = 10

# 처음에는 새로운 batch 2개만 테스트
MAX_NEW_BATCHES = 2

completed_new = 0

for i, start in enumerate(
    range(1, len(shards), BATCH_SIZE),
    start=1
):

    batch_name = f"batch_{i:04d}"

    checkpoint = CHECKPOINTS / f"{batch_name}.json"

    if checkpoint.exists():
        continue

    batch_files = shards[start:start + BATCH_SIZE]

    download_batch(batch_files, batch_name)

    completed_new += 1

    if completed_new >= MAX_NEW_BATCHES:
        break

print("New batches completed:", completed_new)

Downloading: batch_0001


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Completed: batch_0001 | Cells: 2,212
Downloading: batch_0002


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Completed: batch_0002 | Cells: 2,087
New batches completed: 2


In [ ]:
# Whole batch
import time
from datetime import timedelta

BATCH_SIZE = 10

# 전체 batch 목록
all_batches = [
    (f"batch_{i:04d}", shards[start:start + BATCH_SIZE])
    for i, start in enumerate(
        range(1, len(shards), BATCH_SIZE),
        start=1
    )
]

# 이미 완료한 batch 제외
pending = [
    (name, files)
    for name, files in all_batches
    if not (CHECKPOINTS / f"{name}.json").exists()
]

print(f"Remaining batches: {len(pending)}")

start_time = time.perf_counter()

for idx, (batch_name, batch_files) in enumerate(pending, start=1):

    batch_start = time.perf_counter()

    download_batch(batch_files, batch_name)

    batch_time = time.perf_counter() - batch_start

    # 실제 실행 시간 기준으로 남은 시간 추정
    elapsed = time.perf_counter() - start_time
    avg_time = elapsed / idx

    remaining = len(pending) - idx
    eta = avg_time * remaining

    print(
        f"\nProgress: {idx}/{len(pending)}"
        f"\nBatch time: {batch_time:.1f} sec"
        f"\nElapsed: {timedelta(seconds=int(elapsed))}"
        f"\nEstimated remaining: {timedelta(seconds=int(eta))}"
        "\n-----------------------------",
        flush=True
    )

print("\nALL DOWNLOADS COMPLETED!")

Remaining batches: 337
Downloading: batch_0003


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Completed: batch_0003 | Cells: 2,038

Progress: 1/337
Batch time: 14.5 sec
Elapsed: 0:00:14
Estimated remaining: 1:21:11
-----------------------------
Downloading: batch_0004


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Completed: batch_0004 | Cells: 2,014

Progress: 2/337
Batch time: 14.8 sec
Elapsed: 0:00:29
Estimated remaining: 1:21:50
-----------------------------
Downloading: batch_0005


...

Completed: batch_0337 | Cells: 991

Progress: 335/337
Batch time: 9.8 sec
Elapsed: 1:28:13
Estimated remaining: 0:00:31
-----------------------------
Downloading: batch_0338


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Completed: batch_0338 | Cells: 934

Progress: 336/337
Batch time: 10.5 sec
Elapsed: 1:28:24
Estimated remaining: 0:00:15
-----------------------------
Downloading: batch_0339


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Completed: batch_0339 | Cells: 626

Progress: 337/337
Batch time: 5.9 sec
Elapsed: 1:28:30
Estimated remaining: 0:00:00
-----------------------------

ALL DOWNLOADS COMPLETED!


# Check downloaded files

In [ ]:
# Check if download was successful (batches, errors, etc.)
import json
from pathlib import Path
import pyarrow.parquet as pq

PROJECT = Path("/content/drive/MyDrive/Tahoe_project")

PARTS = PROJECT / "raw_parts"
CHECKPOINTS = PROJECT / "checkpoints"

shards = json.loads(
    (PROJECT / "source_shards.json").read_text()
)

# 기존 다운로드 코드와 동일한 batch 구성
expected = {"batch_0000": shards[:1]}

for i, start in enumerate(
    range(1, len(shards), 10),
    start=1
):
    expected[f"batch_{i:04d}"] = shards[start:start+10]

errors = []
total_cells = 0

for name, shard_list in expected.items():

    checkpoint = CHECKPOINTS / f"{name}.json"
    parquet = PARTS / f"{name}.parquet"

    if not checkpoint.exists():
        errors.append(f"Missing checkpoint: {name}")
        continue

    record = json.loads(checkpoint.read_text())

    if record["shards"] != shard_list:
        errors.append(f"Shard mismatch: {name}")
        continue

    n = record["n_cells"]

    if n > 0:

        if not parquet.exists():
            errors.append(f"Missing parquet: {name}")
            continue

        actual = pq.ParquetFile(parquet).metadata.num_rows

        if actual != n:
            errors.append(f"Cell count mismatch: {name}")
            continue

    total_cells += n

print("Expected batches:", len(expected))
print("Verified batches:", len(expected) - len(errors))
print("Total single cells:", f"{total_cells:,}")
print("Errors:", len(errors))

if errors:
    print(errors[:20])
    raise RuntimeError("Download verification failed")
else:
    print("Download verification PASSED!")

Expected batches: 340
Verified batches: 340
Total single cells: 742,008
Errors: 0
Download verification PASSED!


In [ ]:
# Check if all conditions are there
import duckdb
import pandas as pd

con = duckdb.connect()

files = sorted(PARTS.glob("batch_*.parquet"))

paths = [str(f) for f in files]

# Count cells by sample and cell line
counts = con.execute(
    """
    SELECT
        sample,
        cell_line_id,
        COUNT(*) AS n_cells
    FROM read_parquet(?)
    GROUP BY sample, cell_line_id
    """,
    [paths]
).df()

samples = pd.read_csv(
    PROJECT / "selected_samples.csv"
)

samples["is_control"] = (
    samples["is_control"]
    .astype(str)
    .str.lower()
    .eq("true")
)

# Add drug, dose, plate information
df = counts.merge(
    samples[
        ["sample", "drug", "concentration",
         "plate", "is_control"]
    ],
    on="sample",
    how="left",
    validate="many_to_one"
)

assert df["drug"].notna().all(), "Unmatched sample metadata"

# Treatment only
treatment = df[~df["is_control"]].copy()

CELL_IDS = [
    "CVCL_0320",  # HT-29
    "CVCL_0480",  # PANC-1
    "CVCL_1098",  # HepG2/C3A
    "CVCL_0131",  # A-172
    "CVCL_0179",  # BT-474
]

drugs = sorted(treatment["drug"].unique())
doses = [0.05, 0.5, 5.0]

observed = treatment.groupby(
    ["drug", "concentration", "cell_line_id"]
)["n_cells"].sum()

expected_conditions = pd.MultiIndex.from_product(
    [drugs, doses, CELL_IDS],
    names=["drug", "concentration", "cell_line_id"]
)

missing = expected_conditions.difference(
    observed.index
)

print("Total cells:", f"{len(df):,}", "sample-cell-line groups")
print("Compounds:", len(drugs))
print("Expected treatment conditions:", len(expected_conditions))
print("Observed treatment conditions:", len(observed))
print("Missing conditions:", len(missing))

if len(missing):
    print("\nMissing combinations:")
    print(missing.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total cells: 420 sample-cell-line groups
Compounds: 17
Expected treatment conditions: 255
Observed treatment conditions: 255
Missing conditions: 0


In [ ]:
# 17 chemicals X 3 concentrations X 5 cell lines = 255 condition

In [ ]:
# Check DMSO
# Treatment cell counts
treat = (
    df[~df["is_control"]]
    .groupby(
        ["drug", "concentration", "cell_line_id", "plate"],
        as_index=False
    )["n_cells"]
    .sum()
    .rename(columns={"n_cells": "n_treat"})
)

# DMSO cell counts
ctrl = (
    df[df["is_control"]]
    .groupby(
        ["cell_line_id", "plate"],
        as_index=False
    )["n_cells"]
    .sum()
    .rename(columns={"n_cells": "n_dmso"})
)

# Match by cell line and plate
matched = treat.merge(
    ctrl,
    on=["cell_line_id", "plate"],
    how="left",
    validate="many_to_one"
)

# Check missing controls
missing_ctrl = matched[
    matched["n_dmso"].isna()
]

print("Treatment plate-level groups:", len(matched))
print("Groups without matched DMSO:", len(missing_ctrl))

print("\nTreatment cells:")
print(matched["n_treat"].describe())

print("\nMatched DMSO cells:")
print(matched["n_dmso"].describe())

if len(missing_ctrl) > 0:
    print("\nMissing control groups:")
    display(missing_ctrl)
else:
    print("\nAll treatment groups have matched DMSO!")

Treatment plate-level groups: 280
Groups without matched DMSO: 0

Treatment cells:
count     280.000000
mean     1713.196429
std      1347.761875
min       120.000000
25%       823.500000
50%      1368.500000
75%      2140.250000
max      7881.000000
Name: n_treat, dtype: float64

Matched DMSO cells:
count      280.000000
mean      3638.689286
std       2390.922314
min        648.000000
25%       1753.250000
50%       3106.000000
75%       4428.750000
max      10306.000000
Name: n_dmso, dtype: float64

All treatment groups have matched DMSO!
